# Ch 1　為什麼需要 agent 框架：從一次 LLM 呼叫到 agent loop

> 這是 Ch1 的**動手做 notebook**。建議搭配同名 `.md` 章節一起服用：`.md` 講觀念，這裡讓你親手把 agent loop 跑出來。
>
> **需要**：一把 Google (Gemini) API key。把它設成環境變數 `GOOGLE_API_KEY` 即可。
>
> **這一章我們刻意「不用任何 agent 框架」**——只用最原始的 SDK，好讓你親身感受「沒有框架時，你得自己扛下哪些事」。

In [1]:
# 裝一個最素的 LLM SDK——注意，這裡「故意」不裝 langchain / langgraph。
# 本章的精神就是：先體驗手無寸鐵，你才會珍惜後面的框架。
!uv pip install -q google-genai

In [2]:
import os
import getpass

# 沒設金鑰就現場問你，免得你跑到一半才發現忘了設。
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("貼上你的 GOOGLE_API_KEY：")

## 1.1 一次 LLM 呼叫：一個很有自信、但可能在唬爛的純函數

In [3]:
from google import genai

# google-genai 的 Client 會去讀環境變數金鑰；我們明確帶 GOOGLE_API_KEY 最不會搞混。
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# 丟一句進去、吐一段出來。它「聽起來」很篤定——
# 但它根本不知道今天的天氣，這段答案多半是它「編」出來的。
resp = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="台北今天適合戶外活動嗎？",
)
print(resp.text)

要判斷今天台北是否適合戶外活動，建議您可以快速打開手機的天氣 App，並對照以下幾個**關鍵指標**來做決定：

### 1. 快速評估指標

*   **降雨機率：**
    *   **低於 30%**：非常適合戶外活動！
    *   **30% - 60%**：建議攜帶雨具，或選擇「半戶外」的行程（如華山、松菸）。台北常有「午後雷陣雨」，通常集中在下午 2 點到 5 點。
    *   **高於 70%**：建議改成室內行程。
*   **體感溫度與紫外線：**
    *   台北盆地夏天悶熱、冬天濕冷。如果**體感溫度超過 35°C** 或 **紫外線指數（UV）達過量/危險級**，中午前後應避免曝曬，以免中暑。
*   **空氣品質 (AQI)：**
    *   **綠色（良好）/ 黃色（普通）**：適合戶外運動。
    *   **橘色（對敏感族群不健康）以上**：建議過敏體質者減少劇烈戶外運動。

---

### 2. 根據今天天氣的「台北活動推薦」

#### ☀️ 如果今天「晴空萬里 / 舒適陰天」：
這是最棒的戶外活動日！推薦去處：
*   **單車健行：** 大佳河濱公園騎單車、大稻埕碼頭（傍晚看夕陽、貨櫃市集）。
*   **親近自然：** 陽明山（擎天崗、冷水坑）、登象山俯瞰台北 101。
*   **文青散策：** 榕錦時光生活園區、青田街散步。

#### 🥵 如果今天「陽光普照但非常炎熱」：
防曬與防中暑是關鍵，建議選擇「避暑」或「傍晚」行程：
*   **山區避暑：** 陽明山竹子湖、貓空搭纜車喝茶（氣溫會比市區低 2-3 度）。
*   **親水行程：** 內湖大溝溪親水公園、雙溪公園。
*   **夜間活動：** 饒河街或士林夜市、象山看夜景。

#### 🌧️ 如果今天「下雨或有午後雷陣雨」：
建議啟動**雨天備案**（室內或半室內）：
*   **藝文室內：** 台北市立美術館（北美館）、松山文創園區、華山 1914 創意文化園區。
*   **休閒放鬆：** 北投行義路泡溫泉、大稻埕老宅咖啡廳躲雨。
*   **室內運動：** 台北市各區運動中心（羽球、攀岩）、室內溜冰場（小巨蛋滑冰館）。

---

**💡 貼心提醒：**
出門前可以上 **[中央氣象署官網](https://www.cwa.gov.t

## 1.2 工具呼叫：讓模型「開口要」動手

關鍵觀念：模型自己**不會**執行工具，它只會「請求」你幫它呼叫。真正動手的，是你的程式碼。

In [ ]:
from google.genai import types

# 這是給模型看的「工具說明書」——名稱、用途、參數長相。
# 用 FunctionDeclaration「手動」宣告（而非直接丟 Python 函數），模型就不會自動幫你執行，
# 把「要不要呼叫」的決定權留在我們眼前——這正是本章想觀察的。
get_weather_decl = types.FunctionDeclaration(
    name="get_weather",
    description="查詢指定城市的目前天氣",
    parameters={
        "type": "object",
        "properties": {"city": {"type": "string", "description": "城市名，例如 Taipei"}},
        "required": ["city"],
    },
)
config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[get_weather_decl])]
)

resp = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="台北現在天氣如何？",
    config=config,
)

# 模型不直接回答，而是回傳一個 function_call：「我想用工具，結果你給我。」
print("function_calls:", resp.function_calls)

## 1.3 + 1.4　把 reason → act → observe 接成一個真正的 agent loop

下面這 ~40 行，就是一個**完整、能跑的 agent**。請特別留意註解標出的「框架沒幫你、你得自己做」的地方。

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# --- 真正會被「執行」的工具實作（這部分模型碰不到，是你的地盤）---
def get_weather(city: str) -> str:
    fake_db = {"Taipei": "晴，28°C", "Tokyo": "陰，22°C"}   # 先寫死，方便離線示範
    return fake_db.get(city, "查無此城市資料")

# 自己維護一張「工具名稱 -> 實作」的"函數字典"對照表。框架本來會幫你做這件事。
TOOL_IMPLEMENTATIONS = {"get_weather": get_weather}

get_weather_decl = types.FunctionDeclaration(
    name="get_weather",
    description="查詢指定城市的目前天氣",
    parameters={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
)
config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[get_weather_decl])]
)

def run_agent(user_input: str, max_turns: int = 10) -> str:
    # contents 就是對話歷史——每一輪都要自己接好。框架本來會幫你做這件事。
    contents = [types.Content(role="user", parts=[types.Part(text=user_input)])]

    for turn in range(max_turns):                  # max_turns 是護欄：防止它無限要求用工具
        resp = client.models.generate_content(
            model="gemini-2.5-flash", contents=contents, config=config,
        )
        calls = resp.function_calls                 # 模型這一輪要求的工具呼叫（可能多個，或 None）

        if not calls:
            return resp.text                        # 不再要工具 = 任務結束

        # 把模型這一輪的輸出（含 function_call）接回對話。漏了它，下一輪會對不起來。
        contents.append(resp.candidates[0].content)

        # 逐一執行工具，把結果包成 function_response 餵回去（這就是 "observe"）。
        tool_parts = []
        for fc in calls:
            output = TOOL_IMPLEMENTATIONS[fc.name](**dict(fc.args))   # 你的程式負責「執行」（act）
            print(f"  [turn {turn}] 模型要求 {fc.name}({dict(fc.args)}) -> {output}")
            tool_parts.append(
                types.Part.from_function_response(name=fc.name, response={"result": output})
            )
        contents.append(types.Content(role="user", parts=tool_parts))

    return "（達到最大回合數仍未完成）"

print(run_agent("台北和東京現在天氣如何？哪個比較適合戶外活動？"))

## 1.5　數一數：這一趟跑了幾趟 reason → act → observe？

上面的 print 已經幫你標出每一次工具呼叫。回頭看 `[turn N]` 的數量，就是這個 loop 實際轉了幾圈。

**框架完全沒幫你做、而你被迫自己扛的事**：維護 `messages`、dispatch 工具、設停止條件——以及完全缺席的重試、持久化、串流、人工審核。這些「缺口」就是 Part I 之後要一一補上的東西。

In [ ]:
# 練習：把 get_weather 改成有機率丟例外，看看現在這個「裸奔」loop 會怎麼壞。
# 把你「想加但還沒加」的重試邏輯記下來——Ch10 談 fault tolerance 時我們會回來看它。
import random

def flaky_weather(city: str) -> str:
    if random.random() < 0.5:
        raise RuntimeError("天氣 API 暫時掛了 😵")   # 故意製造混亂
    return {"Taipei": "晴，28°C"}.get(city, "查無資料")

# TOOL_IMPLEMENTATIONS["get_weather"] = flaky_weather   # 取消註解就能體驗「沒有重試」的痛
print("把上面那行取消註解、多跑幾次 run_agent，感受一下沒有 fault tolerance 的世界。")